# Phase 3 — Feature Engineering Check
Create lag and rolling window features using PySpark Window functions.
Uses 1% sample to fit in local memory.

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["SPARK_LOCAL_DIRS"] = r"D:\spark-temp"
os.environ["HADOOP_HOME"] = r"D:\hadoop"

os.chdir(r"D:\Retail Demand Forecasting")

In [ ]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales
from retail_demand_forecasting.nodes.feature_engineering import create_features

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("phase3_feature_engineering")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

## 1. Load raw datasets, unpivot, and sample 1%

In [ ]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
calendar_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
sell_prices_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))

melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)
# Sample 1% to fit in local memory
melted_df = melted_df.sample(fraction=0.01, seed=42)
print(f"Melted (1% sample): ~{melted_df.count():,} rows")

## 2. Run create_features node

In [ ]:
params = {
    "lag_days": [7, 28],
    "rolling_window_days": [7, 28],
}

featured_df = create_features(melted_df, params)
print(f"Featured: ~{featured_df.count():,} rows x {len(featured_df.columns)} columns")

## 3. Schema

In [ ]:
featured_df.printSchema()

## 4. Sample rows with features

In [ ]:
featured_df.select(
    "day_id", "date", "item_id", "store_id", "sales",
    "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28",
    "day_of_week", "month", "has_event_1",
).limit(10).toPandas()

## 5. Feature statistics

In [ ]:
featured_df.select(
    "sales", "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28",
).describe().toPandas()

In [ ]:
spark.stop()
print("Phase 3 complete.")